# AURALIS: training the cane's obstacle detector (Neural Network 1)

The cane recognises obstacles with **YOLOv8n**, a convolutional neural network. Out of the box it is trained on COCO (people, vehicles, animals, furniture). This notebook shows how we **teach it new road hazards** with **transfer learning**: we start from the COCO weights and fine-tune on labelled road images on a free Google GPU.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

| Step | What happens | Time on a T4 |
|---|---|---|
| 1 | Install YOLO, check the GPU | 1 min |
| 2 | Download a labelled road-hazard dataset | 1 min |
| 3 | Look at the training images and their boxes | – |
| 4 | Train (transfer learning from COCO) | ~10 min |
| 5 | Results: mAP, precision, recall, confusion matrix, predictions | – |
| 6 | Export for the Raspberry Pi (ONNX, same format the cane already runs) | 1 min |

**Dataset.** With no account, step 2 uses the public *crack-seg* road-damage dataset hosted by Ultralytics (about 4,000 labelled road images), so the notebook always runs. With a free **Roboflow** API key it uses a **pothole** dataset instead (set it in step 2). The final AURALIS model merges pothole, open drain, stairs, curb and other classes with our own cane photos (`ml/merge_datasets.py`); that integration is the next step and is not part of this demo.

In [ ]:
# Step 1: install YOLO and check the GPU
!pip -q install ultralytics roboflow
import torch, ultralytics
print('Ultralytics', ultralytics.__version__, '| PyTorch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch to a T4 GPU runtime (training on CPU takes hours)')

## Step 2: dataset

Leave `ROBOFLOW_API_KEY` empty to use the public road-damage dataset.
For potholes: create a free account on [roboflow.com](https://roboflow.com), open a pothole dataset on [universe.roboflow.com](https://universe.roboflow.com/search?q=pothole) → *Download Dataset* → *YOLOv8* → *Show download code*, and copy the key, workspace, project and version into the cell below.

In [ ]:
# Step 2: download a labelled dataset (YOLO format: one .txt per image, one box per line)
import os, pathlib, yaml, subprocess

ROBOFLOW_API_KEY = ""          # optional: your Roboflow key for the pothole dataset
RF_WORKSPACE, RF_PROJECT, RF_VERSION = "", "", 1

BASE = pathlib.Path('/content') if pathlib.Path('/content').exists() else pathlib.Path.cwd()
if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    ds = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download('yolov8')
    root = pathlib.Path(ds.location)
    names = yaml.safe_load((root / 'data.yaml').read_text())['names']
    cfg = {'path': str(root), 'train': 'train/images', 'val': 'valid/images', 'names': names}
else:
    root = BASE / 'datasets' / 'crack-seg'
    if not root.exists():
        root.mkdir(parents=True, exist_ok=True)
        subprocess.run(['wget', '-q', 'https://github.com/ultralytics/assets/releases/download/v0.0.0/crack-seg.zip',
                        '-O', str(root / 'crack-seg.zip')], check=True)
        subprocess.run(['unzip', '-q', '-o', str(root / 'crack-seg.zip'), '-d', str(root)], check=True)   # -> images/, labels/
    # labels are outlines (polygons); YOLO turns them into boxes for detection
    cfg = {'path': str(root), 'train': 'images/train', 'val': 'images/val', 'names': {0: 'road_crack'}}

DATA_YAML = str(BASE / 'auralis_demo_data.yaml')
pathlib.Path(DATA_YAML).write_text(yaml.safe_dump(cfg))
n_train = len(list((root / cfg['train']).glob('*')))
n_val = len(list((root / cfg['val']).glob('*')))
print(f"classes: {cfg['names']}\ntraining images: {n_train}\nvalidation images: {n_val}")

## Step 3: what the network learns from
Each training image comes with boxes drawn by people. These are the examples the network learns from.

In [ ]:
# Step 3: show some training images with their labelled boxes
import random, cv2, numpy as np, matplotlib.pyplot as plt
random.seed(1)
imgs = sorted((root / cfg['train']).glob('*'))
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, img in zip(axes.flat, random.sample(imgs, 8)):
    im = cv2.cvtColor(cv2.imread(str(img)), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    lbl = pathlib.Path(str(img).replace('/images/', '/labels/')).with_suffix('.txt')
    for line in (lbl.read_text().splitlines() if lbl.exists() else []):
        v = [float(x) for x in line.split()[1:]]
        if len(v) > 4:   # polygon -> box
            xs, ys = v[0::2], v[1::2]
            x1, y1, x2, y2 = min(xs) * w, min(ys) * h, max(xs) * w, max(ys) * h
        else:
            x, y, bw, bh = v
            x1, y1, x2, y2 = (x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (255, 60, 40), max(2, w // 200))
    ax.imshow(im); ax.axis('off')
plt.suptitle('Training images with labelled boxes', fontsize=14); plt.tight_layout(); plt.show()

## Step 4: train (transfer learning)
We start from `yolov8n.pt`, the COCO-trained network the cane already runs, and fine-tune it. The augmentations imitate cane footage: brightness changes, slight rotation from walking, scale changes. Image size 320 px, the same as the cane's fast mode on the Raspberry Pi 4.

In [ ]:
# Step 4: train
from ultralytics import YOLO
EPOCHS = 20          # ~10 min on a T4; use 60+ for the final model
model = YOLO('yolov8n.pt')
model.train(data=DATA_YAML, imgsz=320, epochs=EPOCHS, batch=32, patience=10,
            hsv_v=0.5, degrees=8, translate=0.1, scale=0.4, fliplr=0.5, mosaic=1.0,
            project=str(BASE / 'runs'), name='auralis_demo', exist_ok=True, plots=True)
RUN = BASE / 'runs' / 'auralis_demo'

## Step 5: results
**mAP@0.5** is the standard detection score (1.0 = perfect). **Precision**: of the boxes the network draws, how many are right. **Recall**: of the real hazards, how many it finds.

In [ ]:
# Step 5a: scores on the validation images (never seen during training)
best = YOLO(str(RUN / 'weights' / 'best.pt'))
m = best.val(data=DATA_YAML, imgsz=320, verbose=False)
print(f"mAP@0.5      {m.box.map50:.3f}")
print(f"mAP@0.5:0.95 {m.box.map:.3f}")
print(f"precision    {m.box.mp:.3f}")
print(f"recall       {m.box.mr:.3f}")

In [ ]:
# Step 5b: training curves, precision-recall curve and confusion matrix (for the report)
from IPython.display import Image, display
for pattern in ['results.png', '*PR_curve.png', 'confusion_matrix_normalized.png']:
    for f in sorted(RUN.glob(pattern))[:1]:
        print(f.name); display(Image(str(f), width=900))

In [ ]:
# Step 5c: the trained network on validation images it has never seen
val_imgs = sorted((root / cfg['val']).glob('*'))
random.seed(7)
sample = random.sample(val_imgs, 8)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, r in zip(axes.flat, best.predict([str(p) for p in sample], imgsz=320, conf=0.3, verbose=False)):
    ax.imshow(r.plot()[:, :, ::-1]); ax.axis('off')
plt.suptitle('Predictions of the trained network (validation images)', fontsize=14); plt.tight_layout(); plt.show()

## Step 6: export for the Raspberry Pi
The cane runs networks as **ONNX** with ONNX Runtime (`pi/models/yolov8n.onnx` today). We export the trained network the same way and download it. *(Adding it to the cane is the next step; it is not wired in yet.)*

In [ ]:
# Step 6: export to ONNX (320 px) and download
onnx_path = best.export(format='onnx', imgsz=320)
print('exported', onnx_path)
try:
    from google.colab import files
    files.download(onnx_path)
except ImportError:
    pass